# Grid Infection Simulation Mock Interview

This is a multi-stage interview exercise that can be copied into CoderPad or
run directly in Colab/Jupyter. It focuses on 4-neighbor multi-source BFS,
synchronous updates, per-cell state, off-by-one errors, unit tests, and
randomized differential testing.

> This version uses the four orthogonal neighbors: up, down, left, and right.
> Some variants include diagonal neighbors. Clarify this before coding because
> it changes the propagation time in every test case.

## Running the exercise

```bash
cd algorithm-coding-challenge/grid-infection-puzzle

# Single-file candidate mock; initially fails at the first TODO.
python3 grid_infection_mock.py

# Complete reference implementation.
python3 solution.py

# Colab/Jupyter version.
python3 scripts/generate_notebook.py
python3 scripts/check_notebook.py
```

`grid_infection_mock.py` contains the function signatures, fixed tests, slow
oracles, deterministic random tests, and `run_all_tests()`. The reference
implementation executes 1,400 randomized differential comparisons.

`grid_infection_mock.ipynb` is a self-contained mock interview that runs in
this order:

1. Load common helpers.
2. **Have the candidate write unit tests first**, covering synchronous updates,
   boundary cases, and off-by-one behavior.
3. Implement the seven candidate functions.
4. Load the read-only slow-oracle and randomized-test harness.
5. Run the candidate tests, followed by 1,400 differential comparisons.

The notebook has no third-party dependencies. After completing the TODOs, it
can be run from top to bottom.

## Part 1: Basic Spread

```python
def time_to_full_infection(grid: list[list[int]]) -> int:
    ...
```

- `0` means healthy and `1` means infected.
- At the start of each day, all infected cells simultaneously infect their
  healthy orthogonal neighbors.
- Cells infected today do not propagate until tomorrow.
- Multiple initial infection sources are allowed.
- Return the number of days needed to infect every cell.
- Return `-1` if there is no infection source or any healthy cell is
  unreachable.
- Return `0` for an empty or fully infected grid.

Examples:

- One center source in a 3x3 grid: `2`.
- Two sources in opposite corners of a 3x3 grid: `2`.
- `[[1, 0, 0, 0, 0]]`: `4`.
- `[[1]]`: `0`; `[[0]]`: `-1`.

**Reference approach:** enqueue every initial `1` as a level-0 source and run
multi-source BFS. Each cell is enqueued at most once, so time and space are
both `O(R*C)`.

## Part 2: Immune Cells

```python
def time_to_full_infection_with_immunity(
    grid: list[list[int]],
) -> int:
    ...
```

Add `2 = immune`. An immune cell is permanent, never propagates infection, and
is skipped when counting neighbors, so it behaves like a wall. Return the days
required to infect every non-immune healthy cell, or `-1` if any healthy cell
is permanently isolated. Return `0` if the grid contains only infected and
immune cells.

Some variants encode immune cells as `-1` or use `. / X / I`. Clarify the
encoding before coding. This harness always uses `0 / 1 / 2`.

## Part 3: D-day Recovery to Immunity

```python
def time_to_stable_state(grid: list[list[int]], D: int) -> int:
    ...
```

The input contains only `0` and `1`. After being infected for `D` days, a cell
recovers and becomes immune (`2`). Return the first day with no active infected
cells. Some healthy cells may remain uninfected forever.

### Exact time semantics

`infection_day[r][c]` records the day a cell became infected. Initial sources
have infection day `0`. On each tick:

1. Increment `current_day`.
2. Convert every cell satisfying
   `current_day - infection_day >= D` to immune.
3. A cell that recovered on this tick does **not** spread on the same day.
4. The remaining active infected cells spread from the start-of-day snapshot.
5. A newly infected cell gets `infection_day = current_day` and first spreads
   on the next day.

Therefore, when `D=1`, every initial source recovers before spreading on day 1.
`recovery_oracle()` implements a direct day-by-day simulation. With no initial
walls:

- For `D=1`, the answer is `1` when at least one source exists.
- For `D>=2`, spread time is the 4-neighbor BFS distance, and the answer is
  `last infection day + D`.

## Part 4A: Threshold Infection

```python
def time_to_full_infection_threshold(
    grid: list[list[int]], K: int
) -> int:
    ...
```

Require `1 <= K <= 4`. A healthy cell becomes infected on the next day only if
it has at least `K` infected orthogonal neighbors at the start of the day.
Infected cells never recover, and `2` is an immune wall. Return the number of
days needed to infect every non-immune cell, or `-1` if that cannot happen.

This is no longer ordinary BFS. You can scan the full grid every day, or
maintain an infected-neighbor count and update candidates incrementally from
the current frontier. `threshold_oracle()` provides the slow synchronous
version.

## Part 4B: Death Countdown

Different variants assign different meanings to `K` and death. This mock keeps
two common interpretations in separate functions.

### Countdown version

```python
def time_to_end_with_death_countdown(
    grid: list[list[int]], D: int, K: int, N: int
) -> tuple[int, int]:
    ...
```

- The healthy-cell infection threshold remains `1`.
- Each day processes expired death and recovery deadlines before survivors
  spread.
- An active infected cell starts a countdown when it has at least `K` active
  infected neighbors.
- Its death deadline is `current_day + N` and is not reset if its neighbor
  count later decreases.
- Death wins when death and recovery deadlines occur on the same day.
- If recovery occurs first, the countdown is canceled and the cell becomes
  immune.
- Return `(days until no active infected cells remain, total deaths)`.

### Strict recovery version

```python
def time_to_end_with_death_on_recovery(
    grid: list[list[int]], D: int, K: int
) -> tuple[int, int]:
    ...
```

If a cell has at least `K` active infected neighbors **when it becomes
infected**, mark it as doomed. At its recovery deadline it becomes dead instead
of immune. Initial sources determine their doomed status from the day-0
neighbor snapshot.

## Part 4C: Composite

```python
def time_to_end_composite(
    grid: list[list[int]],
    D: int,
    infection_threshold: int,
    death_threshold: int,
    death_countdown: int,
) -> tuple[int, int]:
    ...
```

Combine immune walls, threshold infection, recovery, and countdown death. This
version must track the active set, infection day, and scheduled death on every
day. Because infected neighbors can recover or die, neighbor counts are not
monotonic and the answer is not simply `last infection time + D`.

## Part 5: Row or Column Burn Optimization

Some interviews continue with: "Each day, choose an entire row or column to
burn and minimize total deaths." This statement is not specific enough to
define a unique problem. Clarify:

- Does burning occur before or after infection spreads?
- How many days can you act, and when does the process stop?
- Do infected and healthy cells have the same death cost?
- Is the goal to prevent all spread, protect specific cells, or eliminate the
  infection?
- Is a cell counted more than once if multiple burns cover it?

Once specified, the problem may become dynamic programming, graph
cut/covering, or time-expanded search. This harness leaves Part 5 as an
interviewer follow-up rather than inventing a false canonical answer.

## BFS versus Simulation

Parts 1-2 are textbook multi-source BFS: each cell is processed once for
`O(R*C)` time. Rescanning the entire grid every day can take
`O((R*C)^2)`. Parts 3-4 add recovery, thresholds, and death states, so they are
usually clearer as synchronous `snapshot -> compute next state -> swap`
simulations. Updating the grid in place while scanning is the most common bug.

## Additional practice variants

| Variant | Rule change | Main skill |
| --- | --- | --- |
| 4-neighbor / 8-neighbor | Parameterize the neighborhood | Contract clarification and parameterized tests |
| Infection time matrix | Return each cell's infection day | BFS distance and unreachable sentinels |
| Final stable grid | Return final state instead of only time | Mutation ownership |
| Multiple infection types | Competing colors and tie-breaking | Event ordering |
| Probabilistic spread | Each edge succeeds randomly | RNG injection and statistical tests |
| Weighted directions | Direction-dependent propagation cost | Dijkstra or 0-1 BFS |
| Terrain effects | Local spread and recovery rules | Separating state from graph structure |
| In-place only | Disallow double buffering | State encoding and day stamps |
| Many queries | Ask for state on arbitrary days | Timeline precomputation |
| Huge grid | Use a sparse frontier | Event-driven simulation and memory layout |


In [ ]:
"""CoderPad-style grid infection interview mock.

Fill the candidate functions and run this file. The harness contains fixed
tests, deterministic random tests, and slow simultaneous-update oracles.
"""

from __future__ import annotations

import random
import sys
from collections.abc import Sequence


Grid = Sequence[Sequence[int]]
DIRS_4 = ((-1, 0), (1, 0), (0, -1), (0, 1))


def _parse_grid(grid: Grid, allowed: set[int]) -> list[list[int]]:
    rows = [list(row) for row in grid]
    if not rows:
        return []
    width = len(rows[0])
    if any(len(row) != width for row in rows):
        raise ValueError("grid must be rectangular")
    if width == 0:
        return []
    invalid = {cell for row in rows for cell in row} - allowed
    if invalid:
        raise ValueError(f"invalid grid states: {sorted(invalid)}")
    return rows


def _neighbors(r: int, c: int, rows: int, cols: int):
    for dr, dc in DIRS_4:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            yield nr, nc

## Your task: write unit tests first

Before implementing the algorithms, write your own unit tests in the next
cell. Use plain `assert` statements and test behavior rather than internal
implementation details.

Cover at least:

- Part 1: empty, no source, already infected, one row/column, multiple sources,
  and a case proving newly infected cells do not spread again on the same day.
- Part 2: a fully blocked healthy region and a partially reachable region.
- Part 3: `D=1` and one case that would expose a recovery/spread off-by-one.
- Part 4: one threshold case and one death/recovery tie.
- Invalid input: one ragged grid or invalid parameter.

Remove the placeholder failure only after adding the tests. The final cell runs
your tests first, followed by the provided deterministic oracle comparisons.


In [ ]:
def run_candidate_unit_tests() -> None:
    # TODO: write your unit tests before implementing the functions below.
    #
    # Example shape only; replace this placeholder with your own assertions:
    # assert time_to_full_infection([[1]]) == 0
    #
    # The provided helper can check invalid inputs:
    # _expect_value_error(time_to_stable_state, [[1]], 0)
    raise AssertionError("TODO: add candidate-written unit tests")


## Implement the challenge

Fill the seven functions below. Run the final test cell after completing a
part; your focused unit tests should fail faster and explain the behavior more
clearly than the full randomized harness.


In [ ]:
def time_to_full_infection(grid: list[list[int]]) -> int:
    """Part 1: return days to infect every cell, or -1 if impossible."""

    raise NotImplementedError


def time_to_full_infection_with_immunity(grid: list[list[int]]) -> int:
    """Part 2: 2 is an immune wall; return -1 if any 0 is unreachable."""

    raise NotImplementedError


def time_to_stable_state(grid: list[list[int]], D: int) -> int:
    """Part 3: recover before spreading when current_day - infection_day >= D."""

    raise NotImplementedError


def time_to_full_infection_threshold(
    grid: list[list[int]], K: int
) -> int:
    """Part 4A: a healthy cell needs at least K infected neighbors."""

    raise NotImplementedError


def time_to_end_with_death_countdown(
    grid: list[list[int]], D: int, K: int, N: int
) -> tuple[int, int]:
    """Part 4B: infection threshold 1 plus an independent death countdown."""

    raise NotImplementedError


def time_to_end_with_death_on_recovery(
    grid: list[list[int]], D: int, K: int
) -> tuple[int, int]:
    """Part 4B strict: surrounded-at-infection cells die at recovery time."""

    raise NotImplementedError


def time_to_end_composite(
    grid: list[list[int]],
    D: int,
    infection_threshold: int,
    death_threshold: int,
    death_countdown: int,
) -> tuple[int, int]:
    """Part 4C: immunity, threshold infection, recovery, and death combined."""

    raise NotImplementedError

## Provided oracle harness

This cell defines deliberately slow synchronous oracles, fixed regression
tests, and deterministic random differential tests. Do not edit it while
solving the challenge.


In [ ]:
# ---------------------------------------------------------------------------
# Slow oracles
# ---------------------------------------------------------------------------
def full_infection_oracle(grid: Grid, *, immune: bool) -> int:
    state = _parse_grid(grid, {0, 1, 2} if immune else {0, 1})
    if not state:
        return 0
    rows, cols = len(state), len(state[0])
    days = 0
    while True:
        healthy = sum(cell == 0 for row in state for cell in row)
        if healthy == 0:
            return days
        newly_infected = [
            (r, c)
            for r in range(rows)
            for c in range(cols)
            if state[r][c] == 0
            and any(
                state[nr][nc] == 1
                for nr, nc in _neighbors(r, c, rows, cols)
            )
        ]
        if not newly_infected:
            return -1
        for r, c in newly_infected:
            state[r][c] = 1
        days += 1


def recovery_oracle(grid: Grid, D: int) -> int:
    if D < 1:
        raise ValueError("D must be positive")
    state = _parse_grid(grid, {0, 1})
    if not state:
        return 0
    rows, cols = len(state), len(state[0])
    infection_day = {
        (r, c): 0
        for r in range(rows)
        for c in range(cols)
        if state[r][c] == 1
    }
    current_day = 0

    while infection_day:
        current_day += 1
        recovering = {
            position
            for position, infected_on in infection_day.items()
            if current_day - infected_on >= D
        }
        for r, c in recovering:
            state[r][c] = 2
            del infection_day[(r, c)]

        active = set(infection_day)
        newly_infected = {
            (nr, nc)
            for r, c in active
            for nr, nc in _neighbors(r, c, rows, cols)
            if state[nr][nc] == 0
        }
        for r, c in newly_infected:
            state[r][c] = 1
            infection_day[(r, c)] = current_day

    return current_day


def threshold_oracle(grid: Grid, K: int) -> int:
    if not 1 <= K <= 4:
        raise ValueError("K must be between 1 and 4")
    state = _parse_grid(grid, {0, 1, 2})
    if not state:
        return 0
    rows, cols = len(state), len(state[0])
    days = 0
    while True:
        healthy = sum(cell == 0 for row in state for cell in row)
        if healthy == 0:
            return days
        newly_infected = []
        for r in range(rows):
            for c in range(cols):
                if state[r][c] != 0:
                    continue
                count = sum(
                    state[nr][nc] == 1
                    for nr, nc in _neighbors(r, c, rows, cols)
                )
                if count >= K:
                    newly_infected.append((r, c))
        if not newly_infected:
            return -1
        for r, c in newly_infected:
            state[r][c] = 1
        days += 1


def composite_oracle(
    grid: Grid,
    D: int,
    infection_threshold: int,
    death_threshold: int,
    death_countdown: int,
) -> tuple[int, int]:
    """Part 4C semantics; Part 4B is the infection_threshold=1 case."""

    if D < 1 or death_countdown < 1:
        raise ValueError("day counts must be positive")
    if not 1 <= infection_threshold <= 4:
        raise ValueError("infection_threshold must be between 1 and 4")
    if not 1 <= death_threshold <= 4:
        raise ValueError("death_threshold must be between 1 and 4")
    state = _parse_grid(grid, {0, 1, 2})
    if not state:
        return 0, 0
    rows, cols = len(state), len(state[0])
    infection_day = [
        [0 if state[r][c] == 1 else None for c in range(cols)]
        for r in range(rows)
    ]
    death_day = [[None] * cols for _ in range(rows)]
    deaths = 0
    current_day = 0

    while any(cell == 1 for row in state for cell in row):
        current_day += 1
        next_state = [row[:] for row in state]
        for r in range(rows):
            for c in range(cols):
                if state[r][c] != 1:
                    continue
                if death_day[r][c] == current_day:
                    next_state[r][c] = 3
                    infection_day[r][c] = None
                    death_day[r][c] = None
                    deaths += 1
                elif current_day - infection_day[r][c] >= D:
                    next_state[r][c] = 2
                    infection_day[r][c] = None
                    death_day[r][c] = None

        counts = [[0] * cols for _ in range(rows)]
        for r in range(rows):
            for c in range(cols):
                if next_state[r][c] != 1:
                    continue
                for nr, nc in _neighbors(r, c, rows, cols):
                    counts[nr][nc] += 1

        for r in range(rows):
            for c in range(cols):
                if (
                    next_state[r][c] == 1
                    and death_day[r][c] is None
                    and counts[r][c] >= death_threshold
                ):
                    death_day[r][c] = current_day + death_countdown
                elif (
                    next_state[r][c] == 0
                    and counts[r][c] >= infection_threshold
                ):
                    next_state[r][c] = 1
                    infection_day[r][c] = current_day

        state = next_state

    return current_day, deaths


def death_countdown_oracle(
    grid: Grid, D: int, K: int, N: int
) -> tuple[int, int]:
    return composite_oracle(grid, D, 1, K, N)


def death_on_recovery_oracle(
    grid: Grid, D: int, K: int
) -> tuple[int, int]:
    if D < 1:
        raise ValueError("D must be positive")
    if not 1 <= K <= 4:
        raise ValueError("K must be between 1 and 4")
    state = _parse_grid(grid, {0, 1, 2})
    if not state:
        return 0, 0
    rows, cols = len(state), len(state[0])
    infection_day = [
        [0 if state[r][c] == 1 else None for c in range(cols)]
        for r in range(rows)
    ]
    doomed = [[False] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            if state[r][c] != 1:
                continue
            count = sum(
                state[nr][nc] == 1
                for nr, nc in _neighbors(r, c, rows, cols)
            )
            doomed[r][c] = count >= K
    deaths = 0
    current_day = 0

    while any(cell == 1 for row in state for cell in row):
        current_day += 1
        next_state = [row[:] for row in state]
        for r in range(rows):
            for c in range(cols):
                if (
                    state[r][c] == 1
                    and current_day - infection_day[r][c] >= D
                ):
                    next_state[r][c] = 3 if doomed[r][c] else 2
                    deaths += doomed[r][c]
                    infection_day[r][c] = None

        counts = [[0] * cols for _ in range(rows)]
        for r in range(rows):
            for c in range(cols):
                if next_state[r][c] != 1:
                    continue
                for nr, nc in _neighbors(r, c, rows, cols):
                    counts[nr][nc] += 1

        for r in range(rows):
            for c in range(cols):
                if next_state[r][c] == 0 and counts[r][c] >= 1:
                    next_state[r][c] = 1
                    infection_day[r][c] = current_day
                    doomed[r][c] = counts[r][c] >= K

        state = next_state

    return current_day, deaths


# ---------------------------------------------------------------------------
# Test harness
# ---------------------------------------------------------------------------


def _expect_equal(actual, expected, label: str) -> None:
    if actual != expected:
        raise AssertionError(f"{label}: got {actual!r}, expected {expected!r}")


def _expect_value_error(fn, *args) -> None:
    try:
        fn(*args)
    except ValueError:
        return
    raise AssertionError(f"{fn.__name__}{args!r} should raise ValueError")


def _fixed_tests(impl) -> None:
    basic_cases = [
        ([], 0),
        ([[]], 0),
        ([[1]], 0),
        ([[0]], -1),
        ([[1, 0, 0, 0, 0]], 4),
        ([[1], [0], [0], [0], [0]], 4),
        ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 2),
        ([[1, 0, 0], [0, 0, 0], [0, 0, 1]], 2),
    ]
    for grid, expected in basic_cases:
        _expect_equal(
            impl.time_to_full_infection(grid), expected, f"Part 1 {grid}"
        )

    immune_cases = [
        ([], 0),
        ([[2]], 0),
        ([[0, 2, 0]], -1),
        ([[1, 0, 2, 0]], -1),
        ([[1, 0], [2, 0]], 2),
        ([[1, 2, 0], [2, 2, 0], [0, 0, 0]], -1),
        ([[1, 0, 2, 0, 0]], -1),
        ([[1, 0, 2, 1, 0]], 1),
    ]
    for grid, expected in immune_cases:
        _expect_equal(
            impl.time_to_full_infection_with_immunity(grid),
            expected,
            f"Part 2 {grid}",
        )

    recovery_cases = [
        ([], 2, 0),
        ([[0, 0]], 2, 0),
        ([[1]], 1, 1),
        ([[1]], 3, 3),
        ([[1, 0]], 1, 1),
        ([[1, 0]], 2, 3),
        ([[1, 0, 0]], 2, 4),
        ([[0, 1, 0], [0, 0, 0]], 3, 5),
    ]
    for grid, D, expected in recovery_cases:
        _expect_equal(
            impl.time_to_stable_state(grid, D),
            expected,
            f"Part 3 {grid}, D={D}",
        )

    threshold_cases = [
        ([], 1, 0),
        ([[0]], 1, -1),
        ([[1, 0, 0]], 1, 2),
        ([[1, 0, 1]], 2, 1),
        ([[1, 0, 0], [1, 0, 0]], 2, -1),
        ([[1, 0, 1], [0, 0, 0]], 2, -1),
        ([[1, 0], [0, 1]], 2, 1),
        ([[1, 2, 0]], 1, -1),
    ]
    for grid, K, expected in threshold_cases:
        _expect_equal(
            impl.time_to_full_infection_threshold(grid, K),
            expected,
            f"Part 4A {grid}, K={K}",
        )

    countdown_cases = [
        ([], 2, 1, 1, (0, 0)),
        ([[1]], 3, 1, 1, (3, 0)),
        ([[1, 1]], 3, 1, 1, (2, 2)),
        ([[1, 1]], 3, 1, 2, (3, 2)),
        ([[1, 1]], 1, 1, 1, (1, 0)),
        ([[1, 1]], 2, 1, 3, (2, 0)),
        ([[1, 0]], 3, 2, 1, (4, 0)),
    ]
    for grid, D, K, N, expected in countdown_cases:
        _expect_equal(
            impl.time_to_end_with_death_countdown(grid, D, K, N),
            expected,
            f"Part 4B countdown {grid}",
        )

    strict_cases = [
        ([], 2, 1, (0, 0)),
        ([[1]], 2, 1, (2, 0)),
        ([[1, 1]], 2, 1, (2, 2)),
        ([[1, 0, 1]], 2, 2, (3, 1)),
        ([[1, 0]], 1, 1, (1, 0)),
    ]
    for grid, D, K, expected in strict_cases:
        _expect_equal(
            impl.time_to_end_with_death_on_recovery(grid, D, K),
            expected,
            f"Part 4B strict {grid}",
        )

    composite_cases = [
        ([], 2, 1, 1, 1, (0, 0)),
        ([[1]], 2, 1, 1, 1, (2, 0)),
        ([[1, 0, 1]], 3, 2, 2, 1, (3, 1)),
        ([[1, 2, 0]], 2, 1, 1, 1, (2, 0)),
    ]
    for grid, D, infection_K, death_K, countdown, expected in composite_cases:
        _expect_equal(
            impl.time_to_end_composite(
                grid, D, infection_K, death_K, countdown
            ),
            expected,
            f"Part 4C {grid}",
        )

    _expect_value_error(impl.time_to_full_infection, [[1, 0], [0]])
    _expect_value_error(impl.time_to_full_infection, [[], [1]])
    _expect_value_error(impl.time_to_full_infection, [[2]])
    _expect_value_error(impl.time_to_full_infection_with_immunity, [[-1]])
    _expect_value_error(impl.time_to_stable_state, [[1]], 0)
    _expect_value_error(impl.time_to_full_infection_threshold, [[1]], 0)
    _expect_value_error(
        impl.time_to_end_with_death_countdown, [[1]], 1, 1, 0
    )


def _random_grid(
    rng: random.Random, rows: int, cols: int, symbols: tuple[int, ...]
) -> list[list[int]]:
    if rows == 0 or cols == 0:
        return []
    return [
        [rng.choice(symbols) for _ in range(cols)]
        for _ in range(rows)
    ]


def _random_tests(impl, seed: int = 20260915) -> None:
    rng = random.Random(seed)
    for case in range(200):
        rows, cols = rng.randint(1, 5), rng.randint(1, 5)
        basic = _random_grid(rng, rows, cols, (0, 0, 0, 1))
        immune = _random_grid(rng, rows, cols, (0, 0, 0, 1, 2, 2))

        _expect_equal(
            impl.time_to_full_infection(basic),
            full_infection_oracle(basic, immune=False),
            f"random Part 1 case {case}",
        )
        _expect_equal(
            impl.time_to_full_infection_with_immunity(immune),
            full_infection_oracle(immune, immune=True),
            f"random Part 2 case {case}",
        )

        D = rng.randint(1, 4)
        _expect_equal(
            impl.time_to_stable_state(basic, D),
            recovery_oracle(basic, D),
            f"random Part 3 case {case}",
        )

        K = rng.randint(1, 4)
        _expect_equal(
            impl.time_to_full_infection_threshold(immune, K),
            threshold_oracle(immune, K),
            f"random Part 4A case {case}",
        )

        N = rng.randint(1, 3)
        _expect_equal(
            impl.time_to_end_with_death_countdown(immune, D, K, N),
            death_countdown_oracle(immune, D, K, N),
            f"random Part 4B countdown case {case}",
        )
        _expect_equal(
            impl.time_to_end_with_death_on_recovery(immune, D, K),
            death_on_recovery_oracle(immune, D, K),
            f"random Part 4B strict case {case}",
        )

        infection_K = rng.randint(1, 4)
        death_K = rng.randint(1, 4)
        _expect_equal(
            impl.time_to_end_composite(immune, D, infection_K, death_K, N),
            composite_oracle(immune, D, infection_K, death_K, N),
            f"random Part 4C case {case}",
        )


def _run_all_tests(impl) -> None:
    _fixed_tests(impl)
    print("PASS fixed tests")
    _random_tests(impl)
    print("PASS 1,400 deterministic random comparisons")


def run_all_tests() -> None:
    _run_all_tests(sys.modules[__name__])

In [ ]:
run_candidate_unit_tests()
run_all_tests()
